# Izza Store - Silver Layer Transformation
## Enterprise Cleansing, Enrichment and Fact Table Build

**Architecture**
```
Store_S001 JSON ---+
Store_S002 JSON    |
Store_S003 JSON    +---> Bronze Delta Tables ---> Silver Tables ---> Gold Marts
Store_S004 JSON    |
Store_S005 JSON ---+
```

| Step | Description |
|------|-------------|
| 1  | Read all Bronze tables (all 5 stores) |
| 2  | Union / merge all stores |
| 3  | Surrogate / composite keys |
| 4  | Customer cleansing + segmentation |
| 5  | Product cleansing + price bands |
| 6  | Staff cleansing + role levels |
| 7  | Orders cleansing + time features |
| 8  | Order Items cleansing + recalculate line_amount |
| 9  | Inventory cleansing + status bands |
| 10 | Cross-table referential integrity + quarantine tables |
| 11 | Business enrichment (KPIs, calendar) |
| 12 | Build silver_fact_sales |
| 13 | Recalculate inventory from actual sales |
| 14 | Save all Silver tables |
| 15 | Gold layer preview (all 15 insights) |
| 16 | Pipeline summary |

> **Runtime:** PySpark on Microsoft Fabric Notebook
> **Target:** Fabric Lakehouse `Tables/silver/`


## Cell 1 - Imports and Spark Session

In [1]:
# -----------------------------------------------------------------------------
# CELL 1 - Imports and Spark Session
#
# Microsoft Fabric provides a pre-configured SparkSession called `spark`.
# All PySpark functions are imported under alias F to avoid name clashes.
# StringType() etc. are used for explicit casts in Silver transformations.
# -----------------------------------------------------------------------------

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, DateType, LongType
)
from datetime import datetime, date

# Fabric provides the session automatically; this is a local-test fallback.
spark = SparkSession.builder.appName('Izza_Silver_Layer').getOrCreate()
spark.conf.set('spark.sql.legacy.timeParserPolicy', 'LEGACY')
spark.conf.set('spark.sql.shuffle.partitions', '8')

# BRONZE_ROOT = "/abfss://new@onelake.dfs.fabric.microsoft.com/test_lakehouse_bronze.Lakehouse/Files/bronze"
# BRONZE_ROOT = "/lakehouse/default/Files/bronze"  # Fabric Lakehouse Files section

BRONZE_ROOT = "Files/bronze"
SILVER_ROOT = 'Tables/silver'   # Fabric Lakehouse Tables section (Delta)

STORE_IDS = ['S001', 'S002', 'S003', 'S004', 'S005']

print('Spark session ready.')
print(f'Bronze root : {BRONZE_ROOT}')
print(f'Silver root : {SILVER_ROOT}')

StatementMeta(, 5958a4d9-a9a7-41e7-8e47-361997473cae, 5, Finished, Available, Finished, False)

Spark session ready.
Bronze root : Files/bronze
Silver root : Tables/silver


---
## Step 1 - Read All Bronze Tables

We use a wildcard glob `*.json` to read ALL batches for ALL stores in one call.
`multiLine=True` handles the pretty-printed JSON from the Bronze notebooks.
`inferSchema=True` is fine at Bronze; types are enforced in Silver.

In [2]:
# -----------------------------------------------------------------------------
# STEP 1 - Read All Bronze Tables
#
# Bronze file naming convention:
#   <table>_<store_id>_<batch_id>.json
# The glob *.json picks up ALL stores and ALL batches in one call.
# For incremental loads, filter by batch_id column after reading.
# -----------------------------------------------------------------------------

def read_bronze(table_name):
    '''
    Read all JSON files for table_name across all stores and batches.
    Returns a single unified DataFrame.
    '''
   
    path = f'{BRONZE_ROOT}/{table_name}/{table_name}_*.json'
    df = (
        spark.read
        .option('multiLine', 'true')
        .option('inferSchema', 'true')
        .json(path)
    )
    print(f'  bronze_{table_name:<15} {df.count():>7,} rows  |  {len(df.columns)} cols')
    return df


print('Reading Bronze tables...')
print('-' * 55)

# bronze_stores      = read_bronze('stores')
bronze_customers   = read_bronze('customers')
bronze_products    = read_bronze('products')
bronze_staff       = read_bronze('staff')
bronze_orders      = read_bronze('orders')
bronze_order_items = read_bronze('order_items')
bronze_inventory   = read_bronze('inventory')

print('-' * 55)
print('All Bronze tables loaded.')

StatementMeta(, 5958a4d9-a9a7-41e7-8e47-361997473cae, 6, Finished, Available, Finished, False)

Reading Bronze tables...
-------------------------------------------------------
  bronze_customers           610 rows  |  11 cols
  bronze_products             40 rows  |  8 cols
  bronze_staff                30 rows  |  8 cols
  bronze_orders            1,366 rows  |  10 cols


  bronze_order_items       3,574 rows  |  9 cols


  bronze_inventory            40 rows  |  10 cols
-------------------------------------------------------
All Bronze tables loaded.


---
## Step 2 - Merge All Stores (Union)

The glob-based Bronze read already unions all 5 stores into each DataFrame.
We alias for cleaner naming. The `store_id` column differentiates S001-S005.

If `store_id` is missing from any table, extract it from the `source_file` column:
```python
F.regexp_extract(F.col('source_file'), r'_(S\d{3})_', 1)
```

In [3]:
# -----------------------------------------------------------------------------
# STEP 2 - Merge All Stores
# The glob-based read already performed the union.
# Alias raw DataFrames for clearer naming downstream.
# -----------------------------------------------------------------------------

customers_raw   = bronze_customers
products_raw    = bronze_products
staff_raw       = bronze_staff
orders_raw      = bronze_orders
order_items_raw = bronze_order_items
inventory_raw   = bronze_inventory
# stores_raw      = bronze_stores

print('Stores present in raw orders:')

orders_raw.groupBy('store_id').count().orderBy('store_id').show()
customers_raw.groupBy('store_id').count().orderBy('store_id').show()

StatementMeta(, 5958a4d9-a9a7-41e7-8e47-361997473cae, 7, Finished, Available, Finished, False)

Stores present in raw orders:


+--------+-----+
|store_id|count|
+--------+-----+
|    S001|  683|
|    S002|  683|
+--------+-----+

+--------+-----+
|store_id|count|
+--------+-----+
|    S001|  305|
|    S002|  305|
+--------+-----+



In [4]:
display(staff_raw.columns)

StatementMeta(, 5958a4d9-a9a7-41e7-8e47-361997473cae, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ea597756-bad3-4e59-bc9d-5982dd540bad)

---
## Step 3 - Surrogate / Composite Keys

Every store starts order IDs from O00001 — raw IDs collide across stores.
Prepending `store_id` creates a globally unique surrogate key.

| Key | Formula |
|-----|---------|
| `order_sk` | `store_id + '_' + order_id` |
| `inventory_sk` | `store_id + '_' + inventory_id` |
| `order_item_sk` | `store_id + '_' + order_id + '_' + product_id` |
| `staff_sk` | `store_id + '_' + staff_id` |

`concat_ws()` is null-safe; `concat()` returns null if any argument is null.

In [5]:
# -----------------------------------------------------------------------------
# STEP 3 - Create Composite Surrogate Keys
# -----------------------------------------------------------------------------

orders_keyed = orders_raw.withColumn(
    'order_sk',
    F.concat_ws('_', F.col('store_id'), F.col('order_id'))
)

inventory_keyed = inventory_raw.withColumn(
    'inventory_sk',
    F.concat_ws('_', F.col('store_id'), F.col('inventory_id'))
)

order_items_keyed = order_items_raw.withColumn(
    'order_item_sk',
    F.concat_ws('_', F.col('store_id'), F.col('order_id'), F.col('product_id'))
)

staff_keyed = staff_raw.withColumn(
    'staff_sk',
    F.concat_ws('_', F.col('store_id'), F.col('staff_id'))
)

total_orders = orders_keyed.count()
distinct_sk  = orders_keyed.select('order_sk').distinct().count()
print(f'Orders total      : {total_orders:,}')
print(f'Distinct order_sk : {distinct_sk:,}')
print(f'Collisions (DQ)   : {total_orders - distinct_sk:,}  <- Bronze duplicates, fixed in Step 7')

StatementMeta(, 5958a4d9-a9a7-41e7-8e47-361997473cae, 9, Finished, Available, Finished, False)

Orders total      : 1,366
Distinct order_sk : 1,360
Collisions (DQ)   : 6  <- Bronze duplicates, fixed in Step 7


---
## Step 4 - Customer Cleansing

**Issues handled:** duplicate records, invalid phone, missing email,
missing city, duplicate email, duplicate phone.

**Transformations:** `trim()`, `initcap()`, `regexp_replace()`, `dropDuplicates()`

**Derived columns added:**
- `customer_age_days` — days since `customer_since`
- `customer_segment` — VIP / Loyal / Regular / Occasional

| Segment | Age Threshold |
|---------|---------------|
| VIP | > 730 days (2 years) |
| Loyal | 365 - 729 days |
| Regular | 180 - 364 days |
| Occasional | < 180 days |

In [6]:
# -----------------------------------------------------------------------------
# STEP 4 - Customer Cleansing
# -----------------------------------------------------------------------------

TODAY = F.current_date()

silver_customers = (
    customers_raw

    # 4.1  Drop exact duplicates by customer_id
    .dropDuplicates(['customer_id'])

    # 4.2  Standardise text fields
    .withColumn('customer_name', F.initcap(F.trim(F.col('customer_name'))))
    .withColumn('city',          F.initcap(F.trim(F.col('city'))))
    .withColumn('email',         F.lower(F.trim(F.col('email'))))

    # 4.3  Phone: strip spaces/dashes, validate 10-digit numeric
    .withColumn('phone_clean', F.regexp_replace(F.col('phone'), r'[\s\-]', ''))
    .withColumn('phone',
        F.when(F.col('phone_clean').rlike(r'^\d{10}$'), F.col('phone_clean'))
         .otherwise(F.lit('INVALID')))
    .drop('phone_clean')

    # 4.4  Null fills
    .withColumn('email',
        F.when(F.col('email').isNull(), F.lit('unknown@noemail.com'))
         .otherwise(F.col('email')))
    .withColumn('city',
        F.when(F.col('city').isNull(), F.lit('Unknown'))
         .otherwise(F.col('city')))

    # 4.5  Cast customer_since to date and derive age in days
    .withColumn('customer_since', F.to_date(F.col('customer_since')))
    .withColumn('customer_age_days',
        F.datediff(TODAY, F.col('customer_since')).cast(IntegerType()))

    # 4.6  Customer segment based on tenure
    .withColumn('customer_segment',
        F.when(F.col('customer_age_days') >= 730, F.lit('VIP'))
         .when(F.col('customer_age_days') >= 365, F.lit('Loyal'))
         .when(F.col('customer_age_days') >= 180, F.lit('Regular'))
         .otherwise(F.lit('Occasional')))

    .withColumn('silver_processed_at', F.current_timestamp())
)

# 4.7  Remove duplicate emails and duplicate phones
#      Keep first occurrence ordered by customer_id
w_email = Window.partitionBy('email').orderBy('customer_id')
w_phone = Window.partitionBy('phone').orderBy('customer_id')

silver_customers = (
    silver_customers
    .withColumn('_email_rank',
        F.when(F.col('email') != 'unknown@noemail.com',
               F.row_number().over(w_email)).otherwise(F.lit(1)))
    .withColumn('_phone_rank',
        F.when(F.col('phone') != 'INVALID',
               F.row_number().over(w_phone)).otherwise(F.lit(1)))
    .filter((F.col('_email_rank') == 1) & (F.col('_phone_rank') == 1))
    .drop('_email_rank', '_phone_rank')
)

print(f'silver_customers : {silver_customers.count():,} rows')
silver_customers.groupBy('customer_segment').count().orderBy('customer_segment').show()

StatementMeta(, 5958a4d9-a9a7-41e7-8e47-361997473cae, 10, Finished, Available, Finished, False)

silver_customers : 299 rows


+----------------+-----+
|customer_segment|count|
+----------------+-----+
|           Loyal|   96|
|      Occasional|   47|
|         Regular|   55|
|             VIP|  101|
+----------------+-----+



---
## Step 5 - Product Cleansing

**Issues handled:** duplicate product_id, category case normalisation
(`pizza / PIZZA / Pizza` → `Pizza`), invalid price (<= 0).

**Derived column:** `price_band`

| Band | Price Range (INR) |
|------|-----------------|
| Budget | <= 60 |
| Mid | 61 - 199 |
| Premium | 200 - 399 |
| Luxury | >= 400 |

In [7]:
# -----------------------------------------------------------------------------
# STEP 5 - Product Cleansing
# -----------------------------------------------------------------------------

silver_products = (
    products_raw

    # 5.1  Deduplicate
    .dropDuplicates(['product_id'])

    # 5.2  Normalise category case: pizza / PIZZA / Pizza -> Pizza
    .withColumn('category',     F.initcap(F.trim(F.lower(F.col('category')))))
    .withColumn('product_name', F.initcap(F.trim(F.col('product_name'))))

    # 5.3  Validate price > 0; null out invalid prices and flag them
    .withColumn('price',
        F.when(F.col('price').cast(DoubleType()) > 0,
               F.col('price').cast(DoubleType()))
         .otherwise(F.lit(None)))
    .withColumn('price_valid_flag',
        F.when(F.col('price').isNull(), F.lit('INVALID')).otherwise(F.lit('VALID')))

    # 5.4  Price band for segment analysis
    .withColumn('price_band',
        F.when(F.col('price') <= 60,  F.lit('Budget'))
         .when(F.col('price') <= 199, F.lit('Mid'))
         .when(F.col('price') <= 399, F.lit('Premium'))
         .otherwise(F.lit('Luxury')))

    .withColumn('silver_processed_at', F.current_timestamp())
)

print(f'silver_products : {silver_products.count():,} rows')
silver_products.select('product_id','product_name','category','price','price_band').show(truncate=False)

StatementMeta(, 5958a4d9-a9a7-41e7-8e47-361997473cae, 11, Finished, Available, Finished, False)

silver_products : 20 rows


+----------+--------------------+--------+-----+----------+
|product_id|product_name        |category|price|price_band|
+----------+--------------------+--------+-----+----------+
|P001      |Margherita Pizza    |Pizza   |299.0|Premium   |
|P002      |Farmhouse Pizza     |Pizza   |399.0|Premium   |
|P003      |Veg Extravaganza    |Pizza   |499.0|Luxury    |
|P004      |Peppy Paneer        |Pizza   |449.0|Luxury    |
|P005      |Garlic Bread        |Sides   |149.0|Mid       |
|P006      |Stuffed Garlic Bread|Sides   |199.0|Mid       |
|P007      |Coke                |Beverage|60.0 |Budget    |
|P008      |Sprite              |Beverage|60.0 |Budget    |
|P009      |Brownie             |Dessert |99.0 |Mid       |
|P010      |Choco Lava Cake     |Dessert |129.0|Mid       |
|P011      |Taco Mexicana       |Sides   |169.0|Mid       |
|P012      |Cheese Dip          |Sides   |49.0 |Budget    |
|P013      |Pasta               |Main    |249.0|Premium   |
|P014      |Veg Burger          |Main   

---
## Step 6 - Staff Cleansing

**Issues handled:** missing `role` → Unknown, invalid `shift` → Unknown.

**Valid shifts:** Morning, Afternoon, Evening. Everything else → Unknown.

**Derived column:** `role_level`

| Role Level | Roles |
|-----------|-------|
| Operations | Manager |
| Frontline | Chef, Cashier, Delivery Executive |
| Support | Cleaner |
| Unknown | Any other |

In [8]:
# -----------------------------------------------------------------------------
# STEP 6 - Staff Cleansing
# -----------------------------------------------------------------------------

VALID_SHIFTS = ['Morning', 'Afternoon', 'Evening']
VALID_ROLES  = ['Manager', 'Chef', 'Cashier', 'Delivery Executive', 'Cleaner']

silver_staff = (
    staff_keyed

    # 6.1  Deduplicate on composite staff_sk
    .dropDuplicates(['staff_sk'])

    # 6.2  Standardise name
    .withColumn('staff_name', F.initcap(F.trim(F.col('staff_name'))))

    # 6.3  Validate role; NULL or unknown values -> 'Unknown'
    .withColumn('role',
        F.when(F.col('role').isin(VALID_ROLES), F.col('role'))
         .otherwise(F.lit('Unknown')))

    # 6.4  Validate shift; Bronze DQ injects 'Night' -> normalise to Unknown
    .withColumn('shift',
        F.when(F.col('shift').isin(VALID_SHIFTS), F.col('shift'))
         .otherwise(F.lit('Unknown')))

    # 6.5  Role level: maps individual roles to organisational tiers
    .withColumn('role_level',
        F.when(F.col('role') == 'Manager',                    F.lit('Operations'))
         .when(F.col('role').isin(['Chef', 'Cashier']),        F.lit('Frontline'))
         .when(F.col('role') == 'Delivery Executive',         F.lit('Frontline'))
         .when(F.col('role') == 'Cleaner',                    F.lit('Support'))
         .otherwise(F.lit('Unknown')))

    .withColumn('silver_processed_at', F.current_timestamp())
)

print(f'silver_staff : {silver_staff.count():,} rows')
silver_staff.groupBy('role','shift','role_level').count().orderBy('role').show()

StatementMeta(, 5958a4d9-a9a7-41e7-8e47-361997473cae, 12, Finished, Available, Finished, False)

silver_staff : 30 rows


+------------------+-------+----------+-----+
|              role|  shift|role_level|count|
+------------------+-------+----------+-----+
|           Cashier|Morning| Frontline|    2|
|           Cashier|Evening| Frontline|    2|
|              Chef|Morning| Frontline|    5|
|              Chef|Evening| Frontline|    5|
|           Cleaner|Evening|   Support|    1|
|           Cleaner|Morning|   Support|    3|
|Delivery Executive|Evening| Frontline|    5|
|Delivery Executive|Unknown| Frontline|    1|
|Delivery Executive|Morning| Frontline|    3|
|           Manager|Morning|Operations|    1|
|           Unknown|Morning|   Unknown|    1|
|           Unknown|Unknown|   Unknown|    1|
+------------------+-------+----------+-----+



---
## Step 7 - Orders Cleansing  *(Most Critical Step)*

**Issues handled:** duplicate order_sk, future timestamp, invalid payment mode,
missing customer_id, missing staff_id, missing store_id.

**Payment mode normalisation:**
PhonePe / Google Pay / Paytm / GPay / BHIM → `UPI`

**Derived columns:**
`order_date`, `order_hour`, `order_month`, `order_day`, `order_weekday`,
`quarter`, `financial_year` (Indian FY Apr-Mar), `week_of_year`,
`peak_hour_flag`, `weekend_flag`, `meal_period`

In [9]:
# -----------------------------------------------------------------------------
# STEP 7 - Orders Cleansing
#
# Most critical step: a corrupt order propagates errors to revenue,
# staff performance, and inventory recalculation downstream.
# -----------------------------------------------------------------------------

VALID_PAYMENT_MODES = ['UPI', 'Cash', 'Card']

# 7.1  Cast order_timestamp to TimestampType
orders_ts = orders_keyed.withColumn(
    'order_timestamp', F.to_timestamp(F.col('order_timestamp'))
)

# 7.2  Payment mode normalisation
#      Real POS systems log UPI app names; fold all back to 'UPI'
orders_norm = orders_ts.withColumn(
    'payment_mode',
    F.when(
        F.col('payment_mode').isin(['PhonePe','Google Pay','Paytm','GPay','BHIM','UPI']),
        F.lit('UPI')
    ).otherwise(F.col('payment_mode'))
)

# 7.3  Flag bad rows for quarantine
orders_flagged = orders_norm.withColumn(
    'reject_reason',
    F.when(F.col('customer_id').isNull(),
           F.lit('MISSING_CUSTOMER_ID'))
     .when(F.col('staff_id').isNull(),
           F.lit('MISSING_STAFF_ID'))
     .when(F.col('store_id').isNull(),
           F.lit('MISSING_STORE_ID'))
     .when(~F.col('payment_mode').isin(VALID_PAYMENT_MODES),
           F.concat_ws(': ', F.lit('INVALID_PAYMENT_MODE'), F.col('payment_mode')))
     .when(F.col('order_timestamp') > F.current_timestamp(),
           F.lit('FUTURE_TIMESTAMP'))
     .otherwise(F.lit(None))
)

# Quarantine rejected orders
silver_rejected_orders = (
    orders_flagged
    .filter(F.col('reject_reason').isNotNull())
    .withColumn('quarantine_at', F.current_timestamp())
)

orders_clean = orders_flagged.filter(F.col('reject_reason').isNull()).drop('reject_reason')

# 7.4  Deduplicate on order_sk; keep earliest ingestion_timestamp (first-write wins)
w_dedup = Window.partitionBy('order_sk').orderBy(F.col('ingestion_timestamp').asc())
orders_deduped = (
    orders_clean
    .withColumn('_rn', F.row_number().over(w_dedup))
    .filter(F.col('_rn') == 1)
    .drop('_rn')
)

# 7.5  Add all time-based derived columns
silver_orders = (
    orders_deduped
    .withColumn('order_date',    F.to_date(F.col('order_timestamp')))
    .withColumn('order_hour',    F.hour(F.col('order_timestamp')))
    .withColumn('order_month',   F.month(F.col('order_timestamp')))
    .withColumn('order_day',     F.dayofmonth(F.col('order_timestamp')))
    .withColumn('order_weekday', F.date_format(F.col('order_timestamp'), 'EEEE'))
    .withColumn('week_of_year',  F.weekofyear(F.col('order_timestamp')))
    .withColumn('quarter',       F.quarter(F.col('order_timestamp')))

    # Indian Financial Year: April-March
    # month >= 4 -> FY = YYYY-(YYYY+1), else FY = (YYYY-1)-YYYY
    .withColumn('financial_year',
        F.when(
            F.month(F.col('order_timestamp')) >= 4,
            F.concat(
                F.year(F.col('order_timestamp')).cast(StringType()), F.lit('-'),
                (F.year(F.col('order_timestamp')) + 1).cast(StringType())
            )
        ).otherwise(
            F.concat(
                (F.year(F.col('order_timestamp')) - 1).cast(StringType()), F.lit('-'),
                F.year(F.col('order_timestamp')).cast(StringType())
            )
        )
    )

    # Peak hours: 19:00 and 20:00 (dinner rush, per Bronze hour weights)
    .withColumn('peak_hour_flag',
        F.when(F.col('order_hour').isin([19, 20]), F.lit('Peak'))
         .otherwise(F.lit('Off-Peak')))

    # Weekend flag
    .withColumn('weekend_flag',
        F.when(F.col('order_weekday').isin(['Saturday','Sunday']), F.lit('Weekend'))
         .otherwise(F.lit('Weekday')))

    # Meal period based on order hour
    .withColumn('meal_period',
        F.when((F.col('order_hour') >= 10) & (F.col('order_hour') < 12), F.lit('Breakfast'))
         .when((F.col('order_hour') >= 12) & (F.col('order_hour') < 15), F.lit('Lunch'))
         .when((F.col('order_hour') >= 15) & (F.col('order_hour') < 17), F.lit('Snack'))
         .when((F.col('order_hour') >= 17) & (F.col('order_hour') < 22), F.lit('Dinner'))
         .otherwise(F.lit('Other')))

    .withColumn('silver_processed_at', F.current_timestamp())
)

print(f'silver_orders          : {silver_orders.count():,} rows')
print(f'silver_rejected_orders : {silver_rejected_orders.count():,} rows')
silver_orders.groupBy('meal_period','peak_hour_flag').count().orderBy('meal_period').show()

StatementMeta(, 5958a4d9-a9a7-41e7-8e47-361997473cae, 13, Finished, Available, Finished, False)

silver_orders          : 716 rows
silver_rejected_orders : 650 rows


+-----------+--------------+-----+
|meal_period|peak_hour_flag|count|
+-----------+--------------+-----+
|  Breakfast|      Off-Peak|   34|
|     Dinner|          Peak|  276|
|     Dinner|      Off-Peak|  196|
|      Lunch|      Off-Peak|  160|
|      Snack|      Off-Peak|   50|
+-----------+--------------+-----+



---
## Step 8 - Order Items Cleansing

**Issues handled:** negative quantity, zero quantity, invalid `product_id` (P999),
invalid `unit_price` (<= 0), duplicate `order_item_sk`.

> **Rule:** `line_amount = quantity * unit_price` is always recalculated.
> Never trust the Bronze value.

In [10]:
# -----------------------------------------------------------------------------
# STEP 8 - Order Items Cleansing
#
# line_amount is ALWAYS recalculated from quantity * unit_price.
# The Bronze value is discarded because Bronze DQ issues (negative qty,
# zero qty) would have propagated incorrect amounts.
# -----------------------------------------------------------------------------

# Collect valid product IDs from the cleansed products table
valid_product_ids = [r.product_id for r in silver_products.select('product_id').collect()]

# 8.1  Flag bad rows for quarantine
items_flagged = order_items_keyed.withColumn(
    'reject_reason',
    F.when(F.col('quantity').cast(IntegerType()) < 0,
           F.lit('NEGATIVE_QUANTITY'))
     .when(F.col('quantity').cast(IntegerType()) == 0,
           F.lit('ZERO_QUANTITY'))
     .when(~F.col('product_id').isin(valid_product_ids),
           F.concat_ws(': ', F.lit('INVALID_PRODUCT_ID'), F.col('product_id')))
     .when(F.col('unit_price').cast(DoubleType()) <= 0,
           F.lit('INVALID_UNIT_PRICE'))
     .otherwise(F.lit(None))
)

silver_rejected_order_items = (
    items_flagged
    .filter(F.col('reject_reason').isNotNull())
    .withColumn('quarantine_at', F.current_timestamp())
)

items_clean = items_flagged.filter(F.col('reject_reason').isNull()).drop('reject_reason')

# 8.2  Deduplicate on order_item_sk; keep earliest ingestion
w_item = Window.partitionBy('order_item_sk').orderBy(F.col('ingestion_timestamp').asc())
items_deduped = (
    items_clean
    .withColumn('_rn', F.row_number().over(w_item))
    .filter(F.col('_rn') == 1)
    .drop('_rn')
)

# 8.3  Cast types and RECALCULATE line_amount
silver_order_items = (
    items_deduped
    .withColumn('quantity',    F.col('quantity').cast(IntegerType()))
    .withColumn('unit_price',  F.col('unit_price').cast(DoubleType()))
    .withColumn('line_amount', F.col('quantity') * F.col('unit_price'))  # recalculated
    .withColumn('silver_processed_at', F.current_timestamp())
)

print(f'silver_order_items          : {silver_order_items.count():,} rows')
print(f'silver_rejected_order_items : {silver_rejected_order_items.count():,} rows')

StatementMeta(, 5958a4d9-a9a7-41e7-8e47-361997473cae, 14, Finished, Available, Finished, False)

silver_order_items          : 3,208 rows
silver_rejected_order_items : 107 rows


---
## Step 9 - Inventory Cleansing

**Issues handled:** negative `closing_stock` (always recalculate), missing
`opening_stock` or `sold_qty` (quarantine), duplicate `inventory_sk`.

> **Rule:** `closing_stock = opening_stock - sold_qty` always recalculated.

**Derived column:** `inventory_status`

| Status | Closing Stock |
|--------|--------------|
| Critical | <= 10 |
| Low Stock | 11 - 50 |
| Medium | 51 - 200 |
| Healthy | > 200 |

In [11]:
# -----------------------------------------------------------------------------
# STEP 9 - Inventory Cleansing
# -----------------------------------------------------------------------------

# 9.1  Flag unresolvable rows (cannot recalculate if inputs are null)
inv_flagged = inventory_keyed.withColumn(
    'reject_reason',
    F.when(F.col('opening_stock').isNull(), F.lit('MISSING_OPENING_STOCK'))
     .when(F.col('sold_qty').isNull(),      F.lit('MISSING_SOLD_QTY'))
     .otherwise(F.lit(None))
)

silver_rejected_inventory = (
    inv_flagged.filter(F.col('reject_reason').isNotNull())
    .withColumn('quarantine_at', F.current_timestamp())
)

inv_clean = inv_flagged.filter(F.col('reject_reason').isNull()).drop('reject_reason')

# 9.2  Deduplicate: keep latest inventory_date per inventory_sk
w_inv = Window.partitionBy('inventory_sk').orderBy(F.col('inventory_date').desc())
inv_deduped = (
    inv_clean
    .withColumn('_rn', F.row_number().over(w_inv))
    .filter(F.col('_rn') == 1)
    .drop('_rn')
)

# 9.3  Cast, recalculate closing_stock, assign inventory_status band
silver_inventory = (
    inv_deduped
    .withColumn('opening_stock', F.col('opening_stock').cast(IntegerType()))
    .withColumn('sold_qty',      F.col('sold_qty').cast(IntegerType()))
    .withColumn('closing_stock', F.col('opening_stock') - F.col('sold_qty'))
    .withColumn('inventory_status',
        F.when(F.col('closing_stock') <= 10,  F.lit('Critical'))
         .when(F.col('closing_stock') <= 50,  F.lit('Low Stock'))
         .when(F.col('closing_stock') <= 200, F.lit('Medium'))
         .otherwise(F.lit('Healthy')))
    .withColumn('silver_processed_at', F.current_timestamp())
)

print(f'silver_inventory          : {silver_inventory.count():,} rows')
print(f'silver_rejected_inventory : {silver_rejected_inventory.count():,} rows')
silver_inventory.groupBy('inventory_status').count().orderBy('inventory_status').show()

StatementMeta(, 5958a4d9-a9a7-41e7-8e47-361997473cae, 15, Finished, Available, Finished, False)

silver_inventory          : 40 rows
silver_rejected_inventory : 0 rows


+----------------+-----+
|inventory_status|count|
+----------------+-----+
|        Critical|    1|
|         Healthy|   38|
|          Medium|    1|
+----------------+-----+



---
## Step 10 - Cross-Table Referential Integrity Validation

**This is where most pipelines fail silently.**

| Check | Source Column | Must Exist In |
|-------|--------------|---------------|
| 1 | orders.customer_id | customers.customer_id |
| 2 | orders.staff_id | staff.staff_id |
| 3 | orders.store_id | stores.store_id |
| 4 | order_items.product_id | products.product_id |
| 5 | order_items.order_id | orders.order_id |

Orphaned records are moved to `silver_rejected_*` quarantine tables.
Clean records are kept for downstream processing.

In [13]:
# -----------------------------------------------------------------------------
# STEP 10 - Cross-Table Referential Integrity
#
# left_anti join returns rows in the LEFT table that have NO match
# in the RIGHT table — these are orphans that must be quarantined.
# -----------------------------------------------------------------------------

valid_customer_ids = silver_customers.select('customer_id')
valid_staff_ids    = silver_staff.select('staff_id')
# valid_store_ids    = stores_raw.select('store_id').dropDuplicates(['store_id'])
valid_prod_ids     = silver_products.select('product_id')
valid_order_ids    = silver_orders.select('order_id')

# 10.1  orders -> customers
orphan_cust = (
    silver_orders.join(valid_customer_ids, on='customer_id', how='left_anti')
    .withColumn('reject_reason', F.lit('ORPHAN_CUSTOMER_ID'))
    .withColumn('quarantine_at', F.current_timestamp())
)
silver_rejected_orders = silver_rejected_orders.unionByName(orphan_cust, allowMissingColumns=True)
silver_orders = silver_orders.join(valid_customer_ids, on='customer_id', how='inner')

# 10.2  orders -> staff
orphan_staff = (
    silver_orders.join(valid_staff_ids, on='staff_id', how='left_anti')
    .withColumn('reject_reason', F.lit('ORPHAN_STAFF_ID'))
    .withColumn('quarantine_at', F.current_timestamp())
)
silver_rejected_orders = silver_rejected_orders.unionByName(orphan_staff, allowMissingColumns=True)
silver_orders = silver_orders.join(valid_staff_ids, on='staff_id', how='inner')

# 10.3  orders -> stores
# orphan_store = (
#     silver_orders.join(valid_store_ids, on='store_id', how='left_anti')
#     .withColumn('reject_reason', F.lit('ORPHAN_STORE_ID'))
#     .withColumn('quarantine_at', F.current_timestamp())
# )
# silver_rejected_orders = silver_rejected_orders.unionByName(orphan_store, allowMissingColumns=True)
# silver_orders = silver_orders.join(valid_store_ids, on='store_id', how='inner')

# 10.4  order_items -> products
orphan_prod = (
    silver_order_items.join(valid_prod_ids, on='product_id', how='left_anti')
    .withColumn('reject_reason', F.lit('ORPHAN_PRODUCT_ID'))
    .withColumn('quarantine_at', F.current_timestamp())
)
silver_rejected_order_items = silver_rejected_order_items.unionByName(orphan_prod, allowMissingColumns=True)
silver_order_items = silver_order_items.join(valid_prod_ids, on='product_id', how='inner')

# 10.5  order_items -> orders
orphan_ord = (
    silver_order_items.join(valid_order_ids, on='order_id', how='left_anti')
    .withColumn('reject_reason', F.lit('ORPHAN_ORDER_ID'))
    .withColumn('quarantine_at', F.current_timestamp())
)
silver_rejected_order_items = silver_rejected_order_items.unionByName(orphan_ord, allowMissingColumns=True)
silver_order_items = silver_order_items.join(valid_order_ids, on='order_id', how='inner')

# 10.6  Customers rejected pool
silver_rejected_customers = (
    customers_raw
    .join(silver_customers.select('customer_id'), on='customer_id', how='left_anti')
    .withColumn('reject_reason', F.lit('DEDUPED_OR_INVALID'))
    .withColumn('quarantine_at', F.current_timestamp())
)

# 10.7  Products rejected pool
silver_rejected_products = (
    products_raw
    .join(silver_products.select('product_id'), on='product_id', how='left_anti')
    .withColumn('reject_reason', F.lit('DEDUPED_OR_INVALID'))
    .withColumn('quarantine_at', F.current_timestamp())
)

print('Cross-table validation complete.')
print(f'  silver_orders (clean)          : {silver_orders.count():,}')
print(f'  silver_rejected_orders         : {silver_rejected_orders.count():,}')
print(f'  silver_order_items (clean)     : {silver_order_items.count():,}')
print(f'  silver_rejected_order_items    : {silver_rejected_order_items.count():,}')
print(f'  silver_rejected_customers      : {silver_rejected_customers.count():,}')
print(f'  silver_rejected_products       : {silver_rejected_products.count():,}')

StatementMeta(, 5958a4d9-a9a7-41e7-8e47-361997473cae, 16, Finished, Available, Finished, False)

Cross-table validation complete.


  silver_orders (clean)          : 1,428


  silver_rejected_orders         : 652


  silver_order_items (clean)     : 3,397
  silver_rejected_order_items    : 189
  silver_rejected_customers      : 2


  silver_rejected_products       : 0


---
## Step 11 - Business Enrichment

Add customer KPIs (`total_orders`, `total_spent`, `avg_order_value`, `last_order_date`)
and store dimension so Gold queries are single-table scans against `silver_fact_sales`.

In [14]:
# -----------------------------------------------------------------------------
# STEP 11 - Business Enrichment
# -----------------------------------------------------------------------------

# 11.1  Customer order counts and last order date
customer_order_counts = (
    silver_orders
    .groupBy('customer_id')
    .agg(
        F.count('order_id').alias('total_orders'),
        F.max('order_date').alias('last_order_date'),
    )
)

# 11.2  Customer revenue aggregates
order_revenue = (
    silver_order_items
    .groupBy('order_id')
    .agg(F.sum('line_amount').alias('order_revenue'))
)
customer_revenue = (
    silver_orders
    .join(order_revenue, on='order_id', how='left')
    .groupBy('customer_id')
    .agg(
        F.sum('order_revenue').alias('total_spent'),
        F.avg('order_revenue').alias('avg_order_value'),
    )
)

# 11.3  Enrich customers with KPIs
silver_customers = (
    silver_customers
    .join(customer_order_counts, on='customer_id', how='left')
    .join(customer_revenue,      on='customer_id', how='left')
    .fillna({'total_orders': 0, 'total_spent': 0.0, 'avg_order_value': 0.0})
)

# 11.4  Store dimension for fact table join
# stores_dim = (
#     stores_raw
#     .select(
#         'store_id',
#         F.col('store_name').alias('store_name'),
#         F.col('store_location').alias('store_location'),
#         F.col('city').alias('store_city'),
#         F.col('state').alias('store_state'),
#         F.col('pincode').alias('store_pincode'),
#     )
#     .dropDuplicates(['store_id'])
# )

print(f'Customer KPIs enriched for {silver_customers.count():,} customers')
silver_customers \
    .select('customer_id','customer_name','customer_segment','total_orders','total_spent') \
    .orderBy(F.col('total_spent').desc()).show(5, truncate=False)

StatementMeta(, 5958a4d9-a9a7-41e7-8e47-361997473cae, 17, Finished, Available, Finished, False)

Customer KPIs enriched for 299 customers


+-----------+---------------+----------------+------------+-----------+
|customer_id|customer_name  |customer_segment|total_orders|total_spent|
+-----------+---------------+----------------+------------+-----------+
|C0003      |Farhan Memon   |Regular         |24          |69424.0    |
|C0036      |Lajita Seshadri|VIP             |20          |64820.0    |
|C0034      |Rishi Minhas   |Loyal           |22          |61530.0    |
|C0041      |Charan Jani    |Loyal           |22          |59658.0    |
|C0022      |Omkaar Chad    |VIP             |16          |52786.0    |
+-----------+---------------+----------------+------------+-----------+
only showing top 5 rows



---
## Step 12 - Build silver_fact_sales

The central denormalised fact table. Grain = one order line item.

```
order_items + orders + customers + products + staff + stores
                      --> silver_fact_sales
```

All Gold layer queries scan this single table with no additional joins.

**Columns (per spec):**
`order_sk, store_id, customer_id, staff_id, product_id, category,`
`quantity, unit_price, line_amount, payment_mode, order_timestamp,`
`order_hour, order_date, customer_segment, store_location, staff_role`

In [15]:
# -----------------------------------------------------------------------------
# STEP 12 - silver_fact_sales
#
# Grain: one row per order line item (order_item_sk).
# All dimension attributes are pre-joined so Gold = simple GROUP BY + SUM.
# -----------------------------------------------------------------------------

# Slim dimension views - only columns needed in the fact table
cust_dim = silver_customers.select(
    'customer_id', 'customer_name',
    F.col('city').alias('customer_city'),
    'customer_segment', 'distance_km'
)

prod_dim = silver_products.select(
    'product_id', 'product_name', 'category', 'price_band'
)

staff_dim = silver_staff.select(
    'staff_id', 'staff_name',
    F.col('role').alias('staff_role'),
    F.col('role_level').alias('staff_role_level')
)

order_dim = silver_orders.select(
    'order_id', 'order_sk', 'store_id', 'customer_id', 'staff_id',
    'payment_mode', 'order_timestamp', 'order_date', 'order_hour',
    'order_month', 'order_day', 'order_weekday', 'week_of_year',
    'quarter', 'financial_year', 'peak_hour_flag', 'weekend_flag', 'meal_period'
)

# Build the fact table step by step
silver_fact_sales = (
    silver_order_items
    .select('order_item_sk', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_amount')

    # Join orders (brings store_id, customer_id, staff_id, all time dimensions)
    .join(order_dim,  on='order_id',    how='inner')

    # Join dimension tables
    .join(cust_dim,   on='customer_id', how='left')
    .join(prod_dim,   on='product_id',  how='left')
    .join(staff_dim,  on='staff_id',    how='left')

    # Canonical column order matching the project spec
    .select(
        'order_item_sk', 'order_sk', 'order_id',
        'store_id',
        'customer_id', 'customer_name', 'customer_city',
        'customer_segment', 'distance_km',
        'staff_id', 'staff_name', 'staff_role', 'staff_role_level',
        'product_id', 'product_name', 'category', 'price_band',
        'quantity', 'unit_price', 'line_amount',
        'payment_mode', 'order_timestamp',
        'order_date', 'order_hour', 'order_day', 'order_month',
        'order_weekday', 'week_of_year', 'quarter', 'financial_year',
        'peak_hour_flag', 'weekend_flag', 'meal_period',
    )
    .withColumn('silver_processed_at', F.current_timestamp())
)

total_facts = silver_fact_sales.count()
total_rev   = silver_fact_sales.agg(F.sum('line_amount')).collect()[0][0]
print(f'silver_fact_sales rows       : {total_facts:,}')
print(f'Total revenue (all stores)   : Rs {total_rev:,.0f}')
silver_fact_sales \
    .groupBy('store_id') \
    .agg(F.count('order_id').alias('lines'), F.sum('line_amount').alias('revenue')) \
    .orderBy('store_id').show()

StatementMeta(, 5958a4d9-a9a7-41e7-8e47-361997473cae, 18, Finished, Available, Finished, False)

silver_fact_sales rows       : 15,784
Total revenue (all stores)   : Rs 7,674,560


+--------+-----+---------+
|store_id|lines|  revenue|
+--------+-----+---------+
|    S001| 2228|1096064.0|
|    S002|13556|6578496.0|
+--------+-----+---------+



---
## Step 13 - Recalculate Inventory from Actual Sales

Instead of trusting Bronze `sold_qty` (approximate at generation time),
we aggregate real quantities from `silver_fact_sales`.

```
sold_qty      = SUM(quantity) per store + product + date  (from silver_fact_sales)
closing_stock = opening_stock - sold_qty
```

Any gap between Bronze and recalculated values shows what DQ quarantine removed.

In [ ]:
# -----------------------------------------------------------------------------
# STEP 13 - Recalculate Inventory from silver_fact_sales
#
# Guarantees inventory is consistent with what was actually sold.
# Gaps highlight records quarantined in earlier steps.
# -----------------------------------------------------------------------------

# Aggregate actual sold quantity per store, product, order_date
actual_sales = (
    silver_fact_sales
    .groupBy('store_id', 'product_id', 'order_date')
    .agg(F.sum('quantity').alias('actual_sold_qty'))
    .withColumnRenamed('store_id',   'fs_store_id')
    .withColumnRenamed('product_id', 'fs_product_id')
)

# Join back to inventory and overwrite sold_qty with actual
silver_inventory_final = (
    silver_inventory
    .join(
        actual_sales,
        on=[
            silver_inventory.store_id   == actual_sales.fs_store_id,
            silver_inventory.product_id == actual_sales.fs_product_id,
        ],
        how='left'
    )
    # Use actual qty when available, fall back to Bronze value
    .withColumn('sold_qty',
        F.coalesce(F.col('actual_sold_qty'), F.col('sold_qty')).cast(IntegerType()))
    # Recalculate closing stock
    .withColumn('closing_stock', F.col('opening_stock') - F.col('sold_qty'))
    # Recalculate status band after correction
    .withColumn('inventory_status',
        F.when(F.col('closing_stock') <= 10,  F.lit('Critical'))
         .when(F.col('closing_stock') <= 50,  F.lit('Low Stock'))
         .when(F.col('closing_stock') <= 200, F.lit('Medium'))
         .otherwise(F.lit('Healthy')))
    .drop('actual_sold_qty', 'order_date', 'fs_store_id', 'fs_product_id')
    .withColumn('silver_processed_at', F.current_timestamp())
)

print(f'silver_inventory_final (recalculated): {silver_inventory_final.count():,} rows')
silver_inventory_final.groupBy('inventory_status').count().orderBy('inventory_status').show()

StatementMeta(, 5958a4d9-a9a7-41e7-8e47-361997473cae, 19, Finished, Available, Finished, False)

silver_inventory_final (recalculated): 40 rows


+----------------+-----+
|inventory_status|count|
+----------------+-----+
|        Critical|   14|
|         Healthy|   20|
|       Low Stock|    1|
|          Medium|    5|
+----------------+-----+



---
## Step 14 - Save All Silver Tables as Delta

All 12 tables written to `Tables/silver/` as Delta format in Fabric Lakehouse.

| Table | Type | Partition |
|-------|------|-----------|
| silver_customers | Core | none |
| silver_products | Core | none |
| silver_staff | Core | none |
| silver_orders | Core | store_id, order_date |
| silver_order_items | Core | store_id |
| silver_inventory | Core | store_id |
| silver_fact_sales | Fact | store_id, order_date |
| silver_rejected_orders | Quarantine | none |
| silver_rejected_customers | Quarantine | none |
| silver_rejected_products | Quarantine | none |
| silver_rejected_inventory | Quarantine | none |
| silver_rejected_order_items | Quarantine | none |

In [ ]:
# -----------------------------------------------------------------------------
# STEP 14 - Save Silver Tables (Delta format, Fabric Lakehouse)
#
# mode='overwrite' + mergeSchema=True -> safe for full daily refresh.
# For incremental / CDC patterns, switch to Delta MERGE (UPSERT) logic.
# partitionBy on store_id + order_date dramatically speeds up Gold queries
# that filter by store or date range.
# -----------------------------------------------------------------------------

def save_silver(df, table_name, partition_cols=None):
    '''
    Write a DataFrame to Fabric Lakehouse Tables/silver/ as Delta.

    Parameters
    ----------
    df            : pyspark DataFrame
    table_name    : str  target table name
    partition_cols: list optional partition columns
    '''
    path = f'{SILVER_ROOT}/{table_name}'
    writer = (
        df.write
        .format('delta')
        .mode('overwrite')
        .option('mergeSchema', 'true')
        .option('overwriteSchema', 'true')
    )
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.save(path)
    print(f'  Saved  {table_name:<40} -> {path}  ({df.count():,} rows)')


print('Saving Silver tables...')
print('-' * 72)

# Core cleansed dimension/fact tables
save_silver(silver_customers,                'silver_customers')
save_silver(silver_products,                 'silver_products')
save_silver(silver_staff,                    'silver_staff')
save_silver(silver_orders,                   'silver_orders',      ['store_id','order_date'])
save_silver(silver_order_items,              'silver_order_items', ['store_id'])
save_silver(silver_inventory_final,          'silver_inventory',   ['store_id'])

# Central fact table (partitioned for fast Gold scans)
save_silver(silver_fact_sales,               'silver_fact_sales',  ['store_id','order_date'])

# Quarantine tables
save_silver(silver_rejected_orders,          'silver_rejected_orders')
save_silver(silver_rejected_customers,       'silver_rejected_customers')
save_silver(silver_rejected_products,        'silver_rejected_products')
save_silver(silver_rejected_inventory,       'silver_rejected_inventory')
save_silver(silver_rejected_order_items,     'silver_rejected_order_items')

print('-' * 72)
print('All Silver tables saved successfully.')

StatementMeta(, 5958a4d9-a9a7-41e7-8e47-361997473cae, 20, Finished, Available, Finished, False)

Saving Silver tables...
------------------------------------------------------------------------


  Saved  silver_customers                         -> Tables/silver/silver_customers  (299 rows)


  Saved  silver_products                          -> Tables/silver/silver_products  (20 rows)


  Saved  silver_staff                             -> Tables/silver/silver_staff  (30 rows)


...

  Saved  silver_fact_sales                        -> Tables/silver/silver_fact_sales  (15,784 rows)


  Saved  silver_rejected_orders                   -> Tables/silver/silver_rejected_orders  (651 rows)


  Saved  silver_rejected_customers                -> Tables/silver/silver_rejected_customers  (2 rows)


  Saved  silver_rejected_products                 -> Tables/silver/silver_rejected_products  (0 rows)
  Saved  silver_rejected_inventory                -> Tables/silver/silver_rejected_inventory  (0 rows)


  Saved  silver_rejected_order_items              -> Tables/silver/silver_rejected_order_items  (189 rows)
------------------------------------------------------------------------
All Silver tables saved successfully.


---
## Step 15 - Gold Layer Preview (All 15 Business Insights)

These validation queries run directly against `silver_fact_sales`
to confirm every Gold insight is reachable without additional joins.

| # | Insight |
|---|---------|
| G01 | Revenue by Hour |
| G02 | Revenue by Product |
| G03 | Revenue by Category |
| G04 | Revenue by Store |
| G05 | Top Customers |
| G06 | Repeat Customers |
| G07 | Average Order Value |
| G08 | Orders by Payment Mode |
| G09 | Orders by Customer Location |
| G10 | Fast Moving Products |
| G11 | Low Stock / Critical Inventory |
| G12 | Revenue per Staff |
| G13 | Orders per Staff |
| G14 | Top Selling Products |
| G15 | Basket Analysis (product pairs) |

In [ ]:
# -----------------------------------------------------------------------------
# STEP 15 - Gold Layer Preview Queries
# Each block = one of the 15 required Gold insights.
# In production, move each to a dedicated Gold notebook.
# -----------------------------------------------------------------------------

print('=' * 62)
print('  GOLD LAYER PREVIEW - All 15 Insights')
print('=' * 62)

# G01 - Revenue by Hour
print('\nG01 - Revenue by Hour')
silver_fact_sales \
    .groupBy('order_hour') \
    .agg(F.sum('line_amount').alias('revenue'), F.count('order_id').alias('lines')) \
    .orderBy('order_hour').show(12)

# G02 - Revenue by Product (Top 10)
print('G02 - Revenue by Product (Top 10)')
silver_fact_sales \
    .groupBy('product_id','product_name','category') \
    .agg(F.sum('line_amount').alias('revenue')) \
    .orderBy(F.col('revenue').desc()).show(10, truncate=False)

# G03 - Revenue by Category
print('G03 - Revenue by Category')
silver_fact_sales \
    .groupBy('category') \
    .agg(F.sum('line_amount').alias('revenue'), F.count('order_id').alias('lines')) \
    .orderBy(F.col('revenue').desc()).show()

# G04 - Revenue by Store
print('G04 - Revenue by Store')
silver_fact_sales \
    .groupBy('store_id') \
    .agg(F.sum('line_amount').alias('revenue'), F.countDistinct('order_id').alias('orders')) \
    .orderBy(F.col('revenue').desc()).show()

# G05 - Top Customers
print('G05 - Top 10 Customers by Lifetime Spend')
silver_fact_sales \
    .groupBy('customer_id','customer_name','customer_segment') \
    .agg(F.sum('line_amount').alias('lifetime_spend'), F.countDistinct('order_id').alias('orders')) \
    .orderBy(F.col('lifetime_spend').desc()).show(10, truncate=False)

# G06 - Repeat Customers
print('G06 - Repeat vs One-time Customers')
silver_fact_sales \
    .groupBy('customer_id').agg(F.countDistinct('order_id').alias('order_count')) \
    .withColumn('customer_type',
        F.when(F.col('order_count') > 1, F.lit('Repeat')).otherwise(F.lit('One-time'))) \
    .groupBy('customer_type').count().show()

# G07 - Average Order Value
print('G07 - Average Order Value by Store')
silver_fact_sales \
    .groupBy('order_id','store_id').agg(F.sum('line_amount').alias('order_value')) \
    .groupBy('store_id') \
    .agg(F.avg('order_value').alias('avg_order_value'), F.count('order_id').alias('orders')) \
    .orderBy('store_id').show()

# G08 - Orders by Payment Mode
print('G08 - Orders by Payment Mode')
silver_fact_sales \
    .groupBy('payment_mode') \
    .agg(F.countDistinct('order_id').alias('orders'), F.sum('line_amount').alias('revenue')) \
    .orderBy(F.col('revenue').desc()).show()

# G09 - Revenue by Customer Location
print('G09 - Revenue by Customer Location (Top 10)')
silver_fact_sales \
    .groupBy('customer_city') \
    .agg(F.countDistinct('order_id').alias('orders'), F.sum('line_amount').alias('revenue')) \
    .orderBy(F.col('revenue').desc()).show(10)

# G10 - Fast Moving Products
print('G10 - Fast Moving Products (by qty sold)')
silver_fact_sales \
    .groupBy('product_id','product_name','category') \
    .agg(F.sum('quantity').alias('total_qty_sold')) \
    .orderBy(F.col('total_qty_sold').desc()).show(10, truncate=False)

# G11 - Low Stock and Critical Inventory
print('G11 - Low Stock and Critical Inventory')
silver_inventory_final \
    .filter(F.col('inventory_status').isin(['Critical','Low Stock'])) \
    .select('store_id','product_id','opening_stock','sold_qty','closing_stock','inventory_status') \
    .orderBy('closing_stock').show(20, truncate=False)

# G12 - Revenue per Staff
print('G12 - Revenue per Staff (Top 10)')
silver_fact_sales \
    .groupBy('staff_id','staff_name','staff_role','store_id') \
    .agg(F.sum('line_amount').alias('revenue_handled')) \
    .orderBy(F.col('revenue_handled').desc()).show(10, truncate=False)

# G13 - Orders per Staff
print('G13 - Orders per Staff (Top 10)')
silver_fact_sales \
    .groupBy('staff_id','staff_name','staff_role') \
    .agg(F.countDistinct('order_id').alias('orders_handled')) \
    .orderBy(F.col('orders_handled').desc()).show(10, truncate=False)

# G14 - Top Selling Products by Revenue
print('G14 - Top Selling Products by Revenue')
silver_fact_sales \
    .groupBy('product_id','product_name','category','price_band') \
    .agg(F.sum('line_amount').alias('revenue'), F.sum('quantity').alias('qty_sold')) \
    .orderBy(F.col('revenue').desc()).show(10, truncate=False)

# G15 - Basket Analysis: top co-purchased product pairs
print('G15 - Basket Analysis: Top Co-purchased Product Pairs')
ba_left  = silver_fact_sales.select('order_id', F.col('product_id').alias('prod_a'), F.col('product_name').alias('name_a'))
ba_right = silver_fact_sales.select('order_id', F.col('product_id').alias('prod_b'), F.col('product_name').alias('name_b'))
ba_left.join(ba_right, on='order_id', how='inner') \
    .filter(F.col('prod_a') < F.col('prod_b')) \
    .groupBy('prod_a','name_a','prod_b','name_b') \
    .agg(F.count('order_id').alias('co_occurrences')) \
    .orderBy(F.col('co_occurrences').desc()).show(10, truncate=False)

print('\nAll 15 Gold insights validated from silver_fact_sales.')

StatementMeta(, 5958a4d9-a9a7-41e7-8e47-361997473cae, 21, Finished, Available, Finished, False)

  GOLD LAYER PREVIEW - All 15 Insights

G01 - Revenue by Hour


+----------+---------+-----+
|order_hour|  revenue|lines|
+----------+---------+-----+
|        10| 183052.0|  388|
|        11| 303072.0|  612|
|        12|1014824.0| 2092|
|        13| 707072.0| 1392|
|        14| 352596.0|  756|
|        15| 159280.0|  340|
|        16| 302644.0|  656|
|        17| 535988.0| 1080|
|        18| 769796.0| 1588|
|        19|1415156.0| 2856|
|        20|1375208.0| 2872|
|        21| 555872.0| 1152|
+----------+---------+-----+

G02 - Revenue by Product (Top 10)


+----------+--------------------+--------+---------+
|product_id|product_name        |category|revenue  |
+----------+--------------------+--------+---------+
|P001      |Margherita Pizza    |Pizza   |1408888.0|
|P002      |Farmhouse Pizza     |Pizza   |1361388.0|
|P004      |Peppy Paneer        |Pizza   |1197932.0|
|P003      |Veg Extravaganza    |Pizza   |1197600.0|
|P013      |Pasta               |Main    |355572.0 |
|P005      |Garlic Bread        |Sides   |272372.0 |
|P006      |Stuffed Garlic Bread|Sides   |267456.0 |
|P010      |Choco Lava Cake     |Dessert |208980.0 |
|P007      |Coke                |Beverage|205440.0 |
|P011      |Taco Mexicana       |Sides   |191984.0 |
+----------+--------------------+--------+---------+
only showing top 10 rows

G03 - Revenue by Category


+--------+---------+-----+
|category|  revenue|lines|
+--------+---------+-----+
|   Pizza|5165808.0| 6556|
|   Sides| 948116.0| 2804|
|    Main| 811036.0| 1760|
|Beverage| 400676.0| 3276|
| Dessert| 348924.0| 1388|
+--------+---------+-----+

G04 - Revenue by Store


...

+----------+--------------------+--------+--------------+
|product_id|product_name        |category|total_qty_sold|
+----------+--------------------+--------+--------------+
|P001      |Margherita Pizza    |Pizza   |4712          |
|P007      |Coke                |Beverage|3436          |
|P002      |Farmhouse Pizza     |Pizza   |3412          |
|P004      |Peppy Paneer        |Pizza   |2692          |
|P003      |Veg Extravaganza    |Pizza   |2400          |
|P008      |Sprite              |Beverage|2068          |
|P005      |Garlic Bread        |Sides   |1828          |
|P010      |Choco Lava Cake     |Dessert |1620          |
|P013      |Pasta               |Main    |1428          |
|P006      |Stuffed Garlic Bread|Sides   |1344          |
+----------+--------------------+--------+--------------+
only showing top 10 rows

G11 - Low Stock and Critical Inventory


+--------+----------+-------------+--------+-------------+----------------+
|store_id|product_id|opening_stock|sold_qty|closing_stock|inventory_status|
+--------+----------+-------------+--------+-------------+----------------+
|S002    |P001      |983          |4096    |-3113        |Critical        |
|S002    |P002      |755          |2848    |-2093        |Critical        |
|S002    |P007      |937          |2872    |-1935        |Critical        |
|S002    |P004      |694          |2292    |-1598        |Critical        |
|S002    |P003      |872          |2064    |-1192        |Critical        |
|S002    |P008      |656          |1796    |-1140        |Critical        |
|S002    |P005      |562          |1644    |-1082        |Critical        |
|S002    |P010      |767          |1380    |-613         |Critical        |
|S002    |P013      |828          |1220    |-392         |Critical        |
|S002    |P006      |868          |1216    |-348         |Critical        |
|S002    |P0

+--------+--------------+------------------+--------+---------------+
|staff_id|staff_name    |staff_role        |store_id|revenue_handled|
+--------+--------------+------------------+--------+---------------+
|ST001   |Onkar Karpe   |Manager           |S002    |287650.0       |
|ST001   |Pushti Handa  |Unknown           |S002    |287650.0       |
|ST009   |Pavani Wali   |Delivery Executive|S002    |273488.0       |
|ST009   |Megha Ganesh  |Delivery Executive|S002    |273488.0       |
|ST004   |Neelima Bhakta|Chef              |S002    |267484.0       |
|ST004   |Arya Atwal    |Chef              |S002    |267484.0       |
|ST005   |Yatin Kota    |Chef              |S002    |229004.0       |
|ST005   |Suhani Thakur |Chef              |S002    |229004.0       |
|ST002   |Omisha Kothari|Chef              |S002    |227844.0       |
|ST002   |Nisha Vig     |Chef              |S002    |227844.0       |
+--------+--------------+------------------+--------+---------------+
only showing top 10 

+----------+--------------------+--------+----------+---------+--------+
|product_id|product_name        |category|price_band|revenue  |qty_sold|
+----------+--------------------+--------+----------+---------+--------+
|P001      |Margherita Pizza    |Pizza   |Premium   |1408888.0|4712    |
|P002      |Farmhouse Pizza     |Pizza   |Premium   |1361388.0|3412    |
|P004      |Peppy Paneer        |Pizza   |Luxury    |1208708.0|2692    |
|P003      |Veg Extravaganza    |Pizza   |Luxury    |1197600.0|2400    |
|P013      |Pasta               |Main    |Premium   |355572.0 |1428    |
|P005      |Garlic Bread        |Sides   |Mid       |272372.0 |1828    |
|P006      |Stuffed Garlic Bread|Sides   |Mid       |267456.0 |1344    |
|P010      |Choco Lava Cake     |Dessert |Mid       |208980.0 |1620    |
|P007      |Coke                |Beverage|Budget    |206160.0 |3436    |
|P011      |Taco Mexicana       |Sides   |Mid       |191984.0 |1136    |
+----------+--------------------+--------+---------

+------+----------------+------+----------------+--------------+
|prod_a|name_a          |prod_b|name_b          |co_occurrences|
+------+----------------+------+----------------+--------------+
|P001  |Margherita Pizza|P007  |Coke            |8624          |
|P001  |Margherita Pizza|P002  |Farmhouse Pizza |8208          |
|P002  |Farmhouse Pizza |P007  |Coke            |7840          |
|P001  |Margherita Pizza|P004  |Peppy Paneer    |6304          |
|P002  |Farmhouse Pizza |P004  |Peppy Paneer    |5312          |
|P001  |Margherita Pizza|P005  |Garlic Bread    |5152          |
|P001  |Margherita Pizza|P008  |Sprite          |4704          |
|P002  |Farmhouse Pizza |P003  |Veg Extravaganza|4576          |
|P003  |Veg Extravaganza|P007  |Coke            |4336          |
|P001  |Margherita Pizza|P003  |Veg Extravaganza|4272          |
+------+----------------+------+----------------+--------------+
only showing top 10 rows


All 15 Gold insights validated from silver_fact_sales.


---
## Step 16 - Pipeline Summary

In [ ]:
# -----------------------------------------------------------------------------
# STEP 16 - Pipeline Summary
# -----------------------------------------------------------------------------

summary_rows = [
    ('silver_customers',             silver_customers.count(),            'Core'),
    ('silver_products',              silver_products.count(),             'Core'),
    ('silver_staff',                 silver_staff.count(),                'Core'),
    ('silver_orders',                silver_orders.count(),               'Core'),
    ('silver_order_items',           silver_order_items.count(),          'Core'),
    ('silver_inventory',             silver_inventory_final.count(),      'Core'),
    ('silver_fact_sales',            silver_fact_sales.count(),           'Fact'),
    ('silver_rejected_orders',       silver_rejected_orders.count(),      'Quarantine'),
    ('silver_rejected_customers',    silver_rejected_customers.count(),   'Quarantine'),
    ('silver_rejected_products',     silver_rejected_products.count(),    'Quarantine'),
    ('silver_rejected_inventory',    silver_rejected_inventory.count(),   'Quarantine'),
    ('silver_rejected_order_items',  silver_rejected_order_items.count(), 'Quarantine'),
]

summary_df = spark.createDataFrame(summary_rows, ['table','rows','type'])

print('\n' + '=' * 62)
print('  IZZA STORE - SILVER LAYER PIPELINE SUMMARY')
print('=' * 62)
summary_df.show(20, truncate=False)

total_rev    = silver_fact_sales.agg(F.sum('line_amount')).collect()[0][0]
total_stores = silver_fact_sales.select('store_id').distinct().count()
print(f'  Stores processed  : {total_stores}')
print(f'  Total revenue     : Rs {total_rev:,.0f}')
print('=' * 62)
print('  Silver layer pipeline complete. All tables saved to Fabric.')

StatementMeta(, 5958a4d9-a9a7-41e7-8e47-361997473cae, 22, Finished, Available, Finished, False)


  IZZA STORE - SILVER LAYER PIPELINE SUMMARY
+---------------------------+-----+----------+
|table                      |rows |type      |
+---------------------------+-----+----------+
|silver_customers           |299  |Core      |
|silver_products            |20   |Core      |
|silver_staff               |30   |Core      |
|silver_orders              |1432 |Core      |
|silver_order_items         |3407 |Core      |
|silver_inventory           |40   |Core      |
|silver_fact_sales          |15832|Fact      |
|silver_rejected_orders     |650  |Quarantine|
|silver_rejected_customers  |2    |Quarantine|
|silver_rejected_products   |0    |Quarantine|
|silver_rejected_inventory  |0    |Quarantine|
|silver_rejected_order_items|189  |Quarantine|
+---------------------------+-----+----------+



  Stores processed  : 2
  Total revenue     : Rs 7,693,220
  Silver layer pipeline complete. All tables saved to Fabric.
